# Copernicus GLO-30 DEM over the Ogallala aquifer at its native 1 arc-second grid, stitched, masked with the aquifer polygon, one GeoTIFF on Drive
Re-creates `GLO_DEM_ogallala.tif` the way it was first made (the provider's 1° tiles, stitched, masked with the aquifer polygon), but in
code, with the checks that were missing and with every Drive export step synchronised before the next one starts.

| | |
|---|---|
| Output | `MyDrive/MSUGWB/GLO_DEM/GLO30_DEM_Ogallala_1arcsec.tif` and `..._manifest.json` (tiles, checksums, statistics, export steps) and `export_steps.jsonl` |
| Source | Copernicus DEM **GLO-30**, 130 tiles of 1° × 1° (3600 × 3600 cells, float32, EPSG:4326), from the public AWS Open Data copy of the product, `https://copernicus-dem-30m.s3.amazonaws.com/` (5.0 GB; no login). The tiles are the product's own files and **are bit-identical to the cells of the earlier QGIS export** (see "Compared with the earlier file") |
| Grid | the product's own: 1/3600° (about 30 m), nothing resampled. Window: the cells whose centre lies inside the bounding box of the aquifer, about 42,913 rows × 34,786 columns (the earlier file has 34,785 columns; the extra east column is empty under the mask) |
| Mask | cells whose centre is outside the aquifer polygon hold the declared **NODATA = −9999** (the earlier file used 0 m, which is a height and declared no nodata). `CLIP_TO_POLYGON = False` keeps every elevation in the bounding box |
| Format | tiled GeoTIFF (512 × 512), DEFLATE with the floating-point predictor, lossless; 1.2 GB instead of 6.0 GB uncompressed |
| Heights | metres above the EGM2008 geoid (as the Copernicus DEM documentation states) |

**Why a re-export.** The data audit of 2026-10-20 found 523,074 cells inside the aquifer that were exactly 0 in the earlier file, 520,796 of them one block
(97.14–96.26° W, 40.64–42.38° N, eastern Nebraska, about 370 km²): missing data stored as a height, in a file with no nodata value. This notebook finds terrain there.

**Compared with the earlier file** (full run on 2026-10-22, cell by cell over the whole aquifer; the earlier file is `merged_datasets/GLO_DEM_ogallala.tif`)
- 614,361,480 cells inside the aquifer have an elevation here: 330.0 to 2395.2 m, mean 982.3 m (the earlier file's mean included the zeros).
- 613,838,400 of them are identical to the earlier file; **none** differs where both hold terrain.
- 523,080 cells are 0 in the earlier file and hold terrain here (the audit counted 523,074 zeros inside the polygon, 520,796 of them the Nebraska block).
- 2,623 cells on the polygon's edge hold a value in the earlier file and NODATA here: the two mask programs (QGIS and this notebook, cell centre inside the polygon) disagree on boundary cells.
- This window has one more column on the east edge (34,786 instead of 34,785); it holds no data under the mask.

**How to run**
1. *Run all* with `TEST_MODE = True`: four tiles in eastern Nebraska (154 MB) into `.../_test/`, every check switched on, and a projection of the whole run.
2. Set `TEST_MODE = False` and *Run all*: 130 tiles (4.7 GB), about 7 GB of scratch disk. On a home PC the whole run took 9 minutes (download 4.5 minutes at 18.6 MB/s, building and checking 4 minutes, six synchronised exports). Running again does nothing if Drive holds the file made from the same tiles (`FORCE = True` builds it again).

**What the notebook does, and checks**
1. *Manifest.* Lists the tiles the window needs (every one must exist in the source's tile list when the aquifer reaches it) and asks the server for each tile's size and ETag (the tile's MD5). Exported to Drive.
2. *Download* up to 8 tiles at a time. Each must have the server's byte count, the MD5 in its ETag, and the header this notebook assumes (3600 × 3600 float32, EPSG:4326, cells centred on whole arc-seconds). Tiles already on scratch that pass are reused.
3. *Stitch and mask.* The output is assembled strip by strip (512 rows, whole rows of blocks, so no block is compressed twice) from the tiles that overlap the strip, by shifts of whole cells. Before a strip is written, **every cell inside the aquifer is checked**: covered by a tile, finite, not NODATA and above 0 m. One failure stops the run and nothing goes to Drive. This is the check the earlier file lacked.
4. *Check the closed file:* header; every strip read back and compared with the checksum taken when it was written; 4,000 random cells looked up by longitude/latitude in the output and in the source tile and compared bit for bit.
5. *Export to Drive*, the GeoTIFF and then the final manifest, each with the synchronisation below.
6. *Final flush:* `drive.flush_and_unmount()`, mount again, and every file of the run is read back from the fresh mount (size and MD5).

**Drive (FUSE) synchronisation: every export step**
A file written to Colab's Drive mount can still be in transit and can read back stale for a while. Each export (manifest, GeoTIFF, final manifest and
each line of the ledger) goes through `drive_put()`: write `<name>.tmp` → `fsync` the file → `fsync` the folder, `os.sync()`, list the folder →
wait `DRIVE_SETTLE_S`, read the `.tmp` back and compare size and MD5 (up to `SYNC_TRIES` times) → rename over the final name → `fsync` the folder again →
(`HARD_FLUSH_EVERY_STEP`) `flush_and_unmount()`, remount and read the final file back once more. Only then is the step logged as synchronised and written to the
ledger. A runtime that dies in the middle leaves the previous good file or a `.tmp` that the next run removes; never a half file under the final name.
`HARD_FLUSH_EVERY_STEP = True` (the default) unmounts and remounts Drive after every step; Colab may ask you to authorise the mount again. Set it to `False` to
do the soft steps after every export and the hard flush only at the end.

**To know**
- The source declares its cells as points (`AREA_OR_POINT=Point`): their centres are on whole arc-seconds and GDAL puts the cell edges half a cell away. The
  output keeps that placement exactly (`cell_centres` tag). The earlier file is on the same lattice.
- GLO-30 is a surface model (buildings, trees); water bodies are flattened by the product. Nothing here edits it.
- Attribution required by the Copernicus DEM licence: "Produced using Copernicus WorldDEM-30 © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018 provided under COPERNICUS by the European Union and ESA; all rights reserved." (also written into the file's `copyright` tag).
- Not tested in Colab itself: the Drive flush and remount are tested here only against stand-ins (the logic and the order of the steps, not Google's mount).

In [ ]:
# ═══════════════ CELL 1: install ═══════════════
# numpy, geopandas, shapely and requests are part of Colab.
# If Colab offers "RESTART RUNTIME" after this cell, restart, then Run all again.
!pip install -q rasterio 2>&1 | tail -2

In [ ]:
# ═══════════════ CELL 2: mount Drive, stage the shapefile, SETTINGS ═══════════════
import os, zipfile
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SHP_ZIP  = '/content/drive/MyDrive/high_plains_quifer.zip'
SHP_DIR  = '/content/ogallala_shp'
SHP_PATH = f'{SHP_DIR}/high_plains_quifer/hp_bound2010.shp'
if not os.path.exists(SHP_PATH):
    with zipfile.ZipFile(SHP_ZIP) as z:
        z.extractall(SHP_DIR)
print("Shapefile ready:", os.path.exists(SHP_PATH))

# ================= SETTINGS (edit these, then Run all) =================
TEST_MODE = True           # True -> rehearsal on a small box into .../_test/ ; False -> the whole bounding box of the aquifer
TEST_BOX  = (-97.9, 40.1, -96.1, 41.9)   # west, south, east, north of the rehearsal: four tiles in eastern Nebraska, the place
                                         # where the earlier export of this DEM held a block of zeros

CLIP_TO_POLYGON = True     # True  -> cells whose centre is outside the aquifer polygon hold NODATA (the earlier export was masked
                           #          with the polygon too, but with 0 m, which is a height; NODATA is declared now)
                           # False -> every cell of the bounding box keeps its elevation (context around the aquifer)
NODATA          = -9999.0  # declared no-data value of the output. Neither GLO-30 nor this export holds it inside the aquifer
FORCE           = False    # True -> build the file again although Drive holds one made from the same tiles

N_DOWNLOAD_THREADS = 8     # tiles downloaded at a time (each 34-45 MB)
STRIP_ROWS         = 512   # rows assembled, checked and written at a time (the block height of the output)
BLOCK_PX           = 512   # internal tile edge of the output GeoTIFF
COMPRESS_LEVEL     = 6     # DEFLATE level of the output (lossless; floating-point predictor)
GDAL_THREADS       = "ALL_CPUS"   # threads GDAL uses to compress the output
POINT_CHECKS       = 4000  # random cells read again straight from the source tiles and compared with the output
KEEP_TILES         = False # True -> keep the downloaded tiles on scratch after the run

MIN_FREE_GB       = 20.0   # scratch space that must stay free at all times
DISK_MAX_WAIT_S   = 2 * 3600
MAX_RETRIES       = 4      # extra download attempts per tile

# ---- Google Drive (FUSE) synchronisation ----
DRIVE_SETTLE_S        = 10     # pause before a file just written to Drive is read back
SYNC_TRIES            = 6      # read-backs (size and MD5) of a file on Drive before the export step is declared failed
HARD_FLUSH_EVERY_STEP = True   # True -> after EVERY export step: flush_and_unmount() + remount + the file is checked again from
                               # the fresh mount (Colab may ask you to authorise the mount again). False -> only after the last step
# =======================================================================

In [ ]:
# ═══════════════ CELL 3: imports, logging, paths, what the source is ═══════════════
import re, gc, sys, json, math, time, zlib, random, shutil, hashlib, logging, threading
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import geopandas as gpd
import shapely
import requests
import rasterio
import rasterio.features
import rasterio.windows
from rasterio.transform import Affine

# force=True: Colab installs its own root log handler, so a plain basicConfig() is a
# no-op there and INFO progress lines would never be shown.
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("glodem")
for _n in ("urllib3.connectionpool", "rasterio", "pyogrio", "fiona"):
    logging.getLogger(_n).setLevel(logging.ERROR)

# ---------- names and paths (Drive = durable, scratch = fast and disposable) ----------
ROI_LABEL   = "Ogallala"
DRIVE_MOUNT = "/content/drive"
DRIVE_OUT   = f"{DRIVE_MOUNT}/MyDrive/MSUGWB/GLO_DEM" + ("/_test" if TEST_MODE else "")
SCRATCH     = "/content/glodem_scratch" + ("_test" if TEST_MODE else "")
TILE_DIR    = f"{SCRATCH}/tiles"          # the downloaded 1-degree tiles
BUILD_DIR   = f"{SCRATCH}/build"          # the output being built, and files waiting to go to Drive
DISK_PATH   = "/content"                  # same filesystem as scratch AND Drive's local upload cache
for _d in (DRIVE_OUT, TILE_DIR, BUILD_DIR):
    os.makedirs(_d, exist_ok=True)

OUT_NAME      = f"GLO30_DEM_{ROI_LABEL}_1arcsec.tif" if not TEST_MODE else f"GLO30_DEM_{ROI_LABEL}_1arcsec_test.tif"
MANIFEST_NAME = OUT_NAME[:-4] + "_manifest.json"       # tiles used (size, ETag), window, settings, checksums of the output
LEDGER_NAME   = "export_steps.jsonl"                   # one line per export step that reached Drive

# ---------- the source: Copernicus DEM GLO-30, as ESA distributes it on AWS Open Data ----------
# 1 x 1 degree Cloud-Optimized GeoTIFFs of 3600 x 3600 cells (1 arc-second, about 30 m in latitude; north of 50 degrees the
# columns are wider, but the aquifer lies south of 44 N), float32, EPSG:4326, heights in metres above the EGM2008 geoid.
# The cells are POINT samples: their CENTRES lie on whole arc-seconds, so their edges are half a cell off the whole-arc-second
# lines (a tile named N40_00_W098_00 holds centres 41.0 N ... 40.00028 N and 98.0 W ... 97.99972 W). Cell (row i, column j) of the
# global grid has its centre at 90 N - i/3600 and -180 + j/3600; tile N<lat> W<lon> starts at row (89 - lat) * 3600, column (lon + 180) * 3600.
# The earlier export of this DEM (stitched from the provider's zip files in QGIS) is on this same lattice.
AWS_BASE     = "https://copernicus-dem-30m.s3.amazonaws.com/"
TILE_PX      = 3600                        # cells per tile edge
CELLS_DEG    = 3600                        # cells per degree
EXPECT_DTYPE = "float32"


def tile_name(lat, lon):
    """Name of the tile whose lower-left corner is (lat, lon), both whole degrees."""
    return f"Copernicus_DSM_COG_10_{'N' if lat >= 0 else 'S'}{abs(lat):02d}_00_{'E' if lon >= 0 else 'W'}{abs(lon):03d}_00_DEM"


def tile_url(name):
    return f"{AWS_BASE}{name}/{name}.tif"


class PipelineStop(RuntimeError):
    """Clean stop: nothing on Drive was changed or only whole, verified steps were; re-running resumes."""


class DataIntegrityError(RuntimeError):
    """Something that must never reach the output was detected."""


def utcnow():
    return datetime.now(timezone.utc).replace(tzinfo=None)


def iso(dt):
    return dt.strftime("%Y-%m-%dT%H:%M:%SZ")

In [ ]:
# ═══════════════ CELL 4: disk floor, and the Drive (FUSE) writer that checks every export step ═══════════════
# Colab's Drive is a FUSE mount: a file that has been written and closed can still be on its way to Drive, can read back with
# stale bytes for a short while, and is only safe once Google has it. EVERY export step of this notebook therefore ends with
# drive_put(), which does, in this order:
#   1  copy to <name>.tmp on Drive, flush Python's buffer and fsync the file
#   2  fsync the Drive folder, os.sync(), list the folder (forces the FUSE metadata to refresh)
#   3  wait DRIVE_SETTLE_S, then read the .tmp file back: size and MD5 must equal the source (up to SYNC_TRIES times)
#   4  rename .tmp over the final name, fsync the folder again, os.sync()
#   5  HARD_FLUSH_EVERY_STEP: drive.flush_and_unmount(), mount again, read the final file back once more from the fresh mount
#   6  only then is the step logged as "synchronised" (and the step recorded in the ledger, itself synchronised the same way)
# A runtime that dies in the middle leaves the previous good file, or a .tmp that the next run removes, never a half file under
# the final name.
TIMES, TOTALS = {}, {}


class timed:
    """with timed("download"): ...  adds the elapsed seconds to TIMES["download"]."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        TIMES[self.name] = TIMES.get(self.name, 0.0) + time.time() - self.t0
        return False


def human(n):
    n = float(n)
    for u in ("B", "KB", "MB", "GB", "TB"):
        if abs(n) < 1024:
            return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"


def free_gb():
    return shutil.disk_usage(DISK_PATH).free / 1024 ** 3


def ensure_disk_free(context="", extra_gb=0.0):
    """Block until MIN_FREE_GB + extra_gb is free. A shortfall normally means Drive's local upload cache is still draining:
    wait for it rather than fail. Raises PipelineStop if it never clears."""
    need = MIN_FREE_GB + max(0.0, extra_gb)
    t0, warned = time.time(), False
    while True:
        free = free_gb()
        if free >= need:
            if warned:
                log.info("Disk OK again: %.1f GB free (need %.1f)", free, need)
            return free
        if not warned:
            warned = True
            log.warning("LOW DISK %.1f < %.1f GB (%s) - waiting for Drive to upload", free, need, context)
        if time.time() - t0 > DISK_MAX_WAIT_S:
            raise PipelineStop(f"Free space stayed at {free:.1f} GB (< {need:.1f} GB) for {DISK_MAX_WAIT_S // 60} min ({context}). "
                               "Nothing is lost - re-run to resume.")
        time.sleep(60)


def md5sum(path, block=16 * 1024 * 1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


# ---------- FUSE synchronisation ----------
SYNC_LOG = []          # (label, file, bytes, md5, seconds, flushed) of every export step that completed


def _fsync_dir(path):
    """fsync a folder so that the new directory entry itself is on its way (not possible on every OS: then a no-op)."""
    try:
        fd = os.open(path, os.O_RDONLY)
    except OSError:
        return
    try:
        os.fsync(fd)
    except OSError:
        pass
    finally:
        os.close(fd)


def soft_sync(folder):
    """Everything short of a remount: folder fsync, sync(), and a listing to make the FUSE layer refresh its view."""
    _fsync_dir(folder)
    if hasattr(os, "sync"):
        os.sync()
    try:
        os.listdir(folder)
    except OSError:
        pass


def hard_flush():
    """Colab's own way to be sure Drive has everything: drive.flush_and_unmount(), then mount again. Outside Colab (a
    rehearsal on a local disk) there is nothing to flush and this says so."""
    try:
        from google.colab import drive
    except ImportError:
        log.info("(not running in Colab: no Drive mount to flush)")
        return False
    drive.flush_and_unmount()
    time.sleep(3)
    drive.mount(DRIVE_MOUNT, force_remount=True)
    for _ in range(30):                                     # the mount point must be back before anything is read
        if os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive")):
            return True
        time.sleep(2)
    raise PipelineStop("Drive did not come back after the flush. Nothing is lost - mount it again and re-run.")


def _read_back_ok(path, size, md5):
    try:
        return os.path.getsize(path) == size and md5sum(path) == md5
    except OSError:
        return False


def drive_put(src, dst, label, hard=None):
    """Put a file on Drive and prove it is there, whole (see the steps in the header of this cell). Raises PipelineStop if
    Drive will not show the right bytes after SYNC_TRIES read-backs; the previous file under `dst` is then untouched."""
    t0 = time.time()
    folder = os.path.dirname(dst)
    os.makedirs(folder, exist_ok=True)
    size, want = os.path.getsize(src), md5sum(src)
    ensure_disk_free(f"Drive copy ({label})", extra_gb=size / 1024 ** 3)       # FUSE caches the upload locally
    tmp = dst + ".tmp"
    last = None
    for attempt in range(1, 4):
        try:
            with open(src, "rb") as fs, open(tmp, "wb") as fd:                    # 1
                shutil.copyfileobj(fs, fd, length=16 * 1024 * 1024)
                fd.flush()
                os.fsync(fd.fileno())
            soft_sync(folder)                                                      # 2
            ok = False
            for _ in range(SYNC_TRIES):                                            # 3
                time.sleep(DRIVE_SETTLE_S)
                if _read_back_ok(tmp, size, want):
                    ok = True
                    break
                soft_sync(folder)
            if not ok:
                raise IOError("Drive shows other bytes than were written (size or MD5), also after waiting")
            os.replace(tmp, dst)                                                   # 4
            soft_sync(folder)
            break
        except Exception as e:
            last = e
            log.warning("Drive export '%s': attempt %d/3 failed: %s", label, attempt, e)
            try:
                os.remove(tmp)
            except OSError:
                pass
            time.sleep(10 * attempt)
    else:
        raise PipelineStop(f"Drive export '{label}' failed after 3 attempts ({last}). The previous file on Drive, if any, is untouched - re-run.")
    flushed = False
    if HARD_FLUSH_EVERY_STEP if hard is None else hard:                           # 5
        flushed = hard_flush()
        if not _read_back_ok(dst, size, want):
            raise PipelineStop(f"Drive export '{label}': after flush_and_unmount and a fresh mount the file does not read back "
                               "identically. Re-run; nothing was recorded for this step.")
    SYNC_LOG.append((label, os.path.basename(dst), size, want, round(time.time() - t0, 1), flushed))
    TIMES["Drive export"] = TIMES.get("Drive export", 0.0) + time.time() - t0
    log.info("FUSE synchronised: %s -> %s (%s, MD5 %s..., %.0f s%s)", label, os.path.basename(dst), human(size), want[:10],
             time.time() - t0, ", flushed and re-read from a fresh mount" if flushed else "")
    return want


def drive_put_text(text, name, label, hard=None):
    """A small text file (JSON, a ledger) through the same steps."""
    local = os.path.join(BUILD_DIR, name + ".upload")
    with open(local, "wb") as f:                      # bytes, so the size on Drive is exactly len(text.encode("utf-8")) on every OS
        f.write(text.encode("utf-8"))
        f.flush()
        os.fsync(f.fileno())
    try:
        return drive_put(local, os.path.join(DRIVE_OUT, name), label, hard)
    finally:
        try:
            os.remove(local)
        except OSError:
            pass


def record_step(label, file, md5, size):
    """Append one line to the ledger on Drive, through the same synchronised steps (the ledger is rewritten whole: it has
    a handful of lines, and an append to a FUSE file is exactly what may be lost)."""
    path = os.path.join(DRIVE_OUT, LEDGER_NAME)
    lines = open(path, "rb").read().decode("utf-8") if os.path.exists(path) else ""
    lines += json.dumps(dict(step=label, file=file, bytes=size, md5=md5, at=iso(utcnow()))) + "\n"
    drive_put_text(lines, LEDGER_NAME, f"ledger after '{label}'")


def clean_drive_leftovers():
    """A .tmp file on Drive is the remains of an export step that did not finish: it is removed, never used."""
    for name in os.listdir(DRIVE_OUT):
        if name.endswith(".tmp"):
            try:
                os.remove(os.path.join(DRIVE_OUT, name))
                log.warning("Removed %s, left on Drive by an interrupted export step.", name)
            except OSError as e:
                log.warning("Could not remove %s (%s).", name, e)

In [ ]:
# ═══════════════ CELL 5: the aquifer polygon, the window on the source's grid, the tile manifest, verified downloads ═══════════════
def load_roi():
    """The aquifer polygon (EPSG:4326, not simplified)."""
    gdf = gpd.read_file(SHP_PATH)
    if gdf.crs is None:
        raise DataIntegrityError("The aquifer shapefile has no CRS.")
    g4 = gdf.to_crs(epsg=4326).geometry
    poly = g4.union_all() if hasattr(g4, "union_all") else g4.unary_union
    if not poly.is_valid:
        poly = poly.buffer(0)
    log.info("Aquifer polygon: %d vertices, bounds lon %.5f to %.5f, lat %.5f to %.5f", shapely.get_num_coordinates(poly), poly.bounds[0], poly.bounds[2], poly.bounds[1], poly.bounds[3])
    return poly


def make_window(poly, box=None):
    """The window of cells to export, on the source's own lattice: the cells whose CENTRE lies inside the bounding box of the
    aquifer (in the rehearsal: TEST_BOX). This is the window of the earlier export (34,785 x 42,913 cells). Indices are global:
    row 0 has its centre at 90 N, column 0 at 180 W (see Cell 3)."""
    w, s, e, n = box or (TEST_BOX if TEST_MODE else poly.bounds)
    gcol0, gcol1 = math.ceil((w + 180) * CELLS_DEG), math.floor((e + 180) * CELLS_DEG) + 1
    grow0, grow1 = math.ceil((90 - n) * CELLS_DEG), math.floor((90 - s) * CELLS_DEG) + 1
    win = dict(grow0=grow0, gcol0=gcol0, h=grow1 - grow0, w=gcol1 - gcol0)
    win["transform"] = Affine(1 / CELLS_DEG, 0.0, -180 + (gcol0 - 0.5) / CELLS_DEG, 0.0, -1 / CELLS_DEG, 90 - (grow0 - 0.5) / CELLS_DEG)
    t = win["transform"]
    win["bounds"] = (t.c, t.f + win["h"] * t.e, t.c + win["w"] * t.a, t.f)             # west, south, east, north cell EDGES
    if box is None:
        log.info("Window: %d rows x %d columns of 1/3600 degree (%s cells); cell centres lon %.6f to %.6f, lat %.6f to %.6f", win["h"], win["w"],
                 f"{win['h'] * win['w']:,}", -180 + gcol0 / CELLS_DEG, -180 + (gcol1 - 1) / CELLS_DEG, 90 - (grow1 - 1) / CELLS_DEG, 90 - grow0 / CELLS_DEG)
    return win


def tiles_for(win):
    """[(lat, lon)] of the 1-degree tiles that hold a cell of the window, north to south, west to east."""
    lons = range(win["gcol0"] // CELLS_DEG - 180, (win["gcol0"] + win["w"] - 1) // CELLS_DEG - 180 + 1)
    lats = range(89 - win["grow0"] // CELLS_DEG, 89 - (win["grow0"] + win["h"] - 1) // CELLS_DEG - 1, -1)
    return [(lat, lon) for lat in lats for lon in lons]


def _get(method, url, **kw):
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            r = requests.request(method, url, timeout=(30, 300), **kw)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            time.sleep(min(60, 5 * attempt))
    raise PipelineStop(f"{url} did not answer: {last}. Nothing is lost - re-run to resume.")


def build_manifest(win, poly):
    """Which tiles are needed and what the server says about each (size, ETag = MD5 of the object, Last-Modified)."""
    listed = set(_get("GET", AWS_BASE + "tileList.txt").text.split())
    tiles, absent = [], []
    for lat, lon in tiles_for(win):
        name = tile_name(lat, lon)
        if name not in listed:
            if shapely.box(lon, lat, lon + 1, lat + 1).intersects(poly) and not TEST_MODE:
                raise PipelineStop(f"{name} is not in the source's tile list, and the aquifer reaches it.")
            absent.append(name)
            continue
        tiles.append(dict(name=name, lat=lat, lon=lon, url=tile_url(name)))

    def head(t):
        r = _get("HEAD", t["url"])
        t.update(size=int(r.headers["Content-Length"]), etag=(r.headers.get("ETag") or "").strip('"'), last_modified=r.headers.get("Last-Modified", ""))
        return t

    with ThreadPoolExecutor(6) as ex:
        tiles = list(ex.map(head, tiles))
    no_md5 = [t["name"] for t in tiles if not re.fullmatch(r"[0-9a-f]{32}", t["etag"])]
    if no_md5:
        log.warning("%d tile(s) have an ETag that is not an MD5 (multipart upload); their size is checked, their content only by opening them.", len(no_md5))
    if absent:
        log.info("%d tile(s) of the window do not exist at the source (no land there): %s", len(absent), absent[:3])
    log.info("%d tiles to download, %s", len(tiles), human(sum(t["size"] for t in tiles)))
    return tiles, absent


def check_tile(path, t):
    """The header of a downloaded tile must be exactly what the whole export assumes."""
    with rasterio.open(path) as ds:
        tr = ds.transform
        if (ds.count != 1 or ds.width != TILE_PX or ds.height != TILE_PX or ds.dtypes[0] != EXPECT_DTYPE or str(ds.crs) != "EPSG:4326"
                or abs(tr.a - 1 / CELLS_DEG) > 1e-12 or abs(tr.e + 1 / CELLS_DEG) > 1e-12 or tr.b or tr.d
                or abs(tr.c - (t["lon"] - 0.5 / CELLS_DEG)) > 1e-9 or abs(tr.f - (t["lat"] + 1 + 0.5 / CELLS_DEG)) > 1e-9):
            raise DataIntegrityError(f"{t['name']} is not the 3600 x 3600 float32 EPSG:4326 tile of 1/3600 degree cells whose first cell is centred on "
                                     f"{t['lon']} E, {t['lat'] + 1} N that this notebook assumes: {ds.width}x{ds.height} {ds.dtypes} {ds.crs} {tr}")
        return dict(nodata=ds.nodata, area_or_point=ds.tags().get("AREA_OR_POINT"))


def download_tile(t):
    """One tile: byte count equals the server's, MD5 equals its ETag, header as assumed. A tile already on scratch that passes
    the same checks is kept."""
    dest = os.path.join(TILE_DIR, t["name"] + ".tif")
    part = dest + ".part"

    def good(p):
        return os.path.getsize(p) == t["size"] and (not re.fullmatch(r"[0-9a-f]{32}", t["etag"]) or md5sum(p) == t["etag"])

    if os.path.exists(dest):
        try:
            if good(dest):
                return dict(t, **check_tile(dest, t), path=dest, reused=True)
        except (OSError, rasterio.errors.RasterioError):
            pass
        os.remove(dest)
    last = None
    for attempt in range(1, MAX_RETRIES + 2):
        try:
            with requests.get(t["url"], stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                with open(part, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        if chunk:
                            f.write(chunk)
                    f.flush()
                    os.fsync(f.fileno())
            if not good(part):
                raise IOError("size or MD5 differs from the server's")
            info = check_tile(part, t)
            os.replace(part, dest)
            return dict(t, **info, path=dest, reused=False)
        except DataIntegrityError:
            raise
        except Exception as e:
            last = e
            try:
                os.remove(part)
            except OSError:
                pass
            log.warning("download %s attempt %d/%d: %s", t["name"], attempt, MAX_RETRIES + 1, e)
            time.sleep(min(60, 5 * attempt))
    raise PipelineStop(f"{t['name']} could not be downloaded after {MAX_RETRIES + 1} attempts ({last}). Nothing is lost - re-run to resume.")


def download_all(tiles):
    with timed("download"):
        ensure_disk_free("download", extra_gb=sum(t["size"] for t in tiles) / 1024 ** 3 * 1.6)   # tiles + the output being built
        t0, done, out = time.time(), 0, []
        with ThreadPoolExecutor(N_DOWNLOAD_THREADS) as ex:
            futs = [ex.submit(download_tile, t) for t in tiles]
            for f in as_completed(futs):
                out.append(f.result())
                done += 1
                if done % 20 == 0 or done == len(tiles):
                    log.info("%d/%d tiles on scratch (%s)", done, len(tiles), human(sum(x["size"] for x in out)))
        nbytes = sum(x["size"] for x in out if not x["reused"])
        dt = max(1e-6, time.time() - t0)
        log.info("Downloaded %d tile(s), %s in %.0f s (%.1f MB/s); %d already on scratch", sum(1 for x in out if not x["reused"]),
                 human(nbytes), dt, nbytes / 1e6 / dt, sum(1 for x in out if x["reused"]))
        TOTALS["bytes_downloaded"] = nbytes
    nod = {x["nodata"] for x in out}
    if nod != {None}:
        log.warning("Some tiles declare a no-data value %s; this notebook treats every finite value as elevation and stops on NODATA %s inside the aquifer.", nod, NODATA)
    return sorted(out, key=lambda x: (-x["lat"], x["lon"]))

In [ ]:
# ═══════════════ CELL 6: assemble the output strip by strip, checking every cell of the aquifer ═══════════════
def assemble_strip(handles, tiles, win, r0, nr):
    """Rows r0..r0+nr of the window as float32, copied from the tiles that overlap them (whole-cell shifts: the tiles and the
    window share one lattice, so nothing is resampled). Cells no tile covers hold NODATA. Returns (strip, covered)."""
    strip = np.full((nr, win["w"]), NODATA, dtype="f4")
    covered = np.zeros((nr, win["w"]), dtype=bool)
    g0 = win["grow0"] + r0
    for t in tiles:
        tr0, tc0 = (89 - t["lat"]) * CELLS_DEG, (t["lon"] + 180) * CELLS_DEG          # global row / column of the tile's first cell
        rr0, rr1 = max(tr0, g0), min(tr0 + TILE_PX, g0 + nr)
        cc0, cc1 = max(tc0, win["gcol0"]), min(tc0 + TILE_PX, win["gcol0"] + win["w"])
        if rr0 >= rr1 or cc0 >= cc1:
            continue
        a = handles[t["name"]].read(1, window=rasterio.windows.Window(cc0 - tc0, rr0 - tr0, cc1 - cc0, rr1 - rr0))
        strip[rr0 - g0:rr1 - g0, cc0 - win["gcol0"]:cc1 - win["gcol0"]] = a
        covered[rr0 - g0:rr1 - g0, cc0 - win["gcol0"]:cc1 - win["gcol0"]] = True
    return strip, covered


def build_output(path, win, tiles, poly):
    """Write the GeoTIFF on scratch: tiled, DEFLATE with the floating-point predictor, one write per block (the strips are whole
    rows of blocks, so no block is compressed twice). Every strip is checked before it is written: inside the aquifer a cell that
    is not covered by a tile, is NaN, is NODATA or is not above 0 m stops the run (that is how the earlier export's block of
    zeros would have been caught). Returns the statistics and the checksum of every strip."""
    h, w = win["h"], win["w"]
    profile = dict(driver="GTiff", dtype=EXPECT_DTYPE, count=1, width=w, height=h, crs="EPSG:4326", transform=win["transform"], nodata=NODATA,
                   tiled=True, blockxsize=BLOCK_PX, blockysize=BLOCK_PX, compress="DEFLATE", zlevel=COMPRESS_LEVEL, predictor=3,
                   BIGTIFF="IF_SAFER", NUM_THREADS=GDAL_THREADS)
    ensure_disk_free("build", extra_gb=h * w * 4 / 1024 ** 3 * 0.6)
    st = dict(n_in=0, sum_in=0.0, min_in=np.inf, max_in=-np.inf, n_all=0, min_all=np.inf, max_all=-np.inf, bad=0, crcs={})
    handles = {t["name"]: rasterio.open(t["path"]) for t in tiles}
    t0 = time.time()
    try:
        with rasterio.open(path, "w", **profile) as out:
            for r0 in range(0, h, STRIP_ROWS):
                nr = min(STRIP_ROWS, h - r0)
                strip, covered = assemble_strip(handles, tiles, win, r0, nr)
                if poly is not None:
                    tr = win["transform"] * Affine.translation(0, r0)
                    inside = rasterio.features.geometry_mask([poly], out_shape=(nr, w), transform=tr, invert=True, all_touched=False)
                else:
                    inside = np.zeros((nr, w), dtype=bool)
                good = covered & np.isfinite(strip) & (strip != NODATA)
                bad = inside & (~good | (strip <= 0))
                if bad.any():
                    rr, cc = np.argwhere(bad)[0]
                    why = ("not covered by any tile" if not covered[rr, cc] else "NaN" if not np.isfinite(strip[rr, cc]) else
                           "NODATA" if strip[rr, cc] == NODATA else f"{strip[rr, cc]:.3f} m, not above 0")
                    raise DataIntegrityError(f"{int(bad.sum())} cell(s) inside the aquifer have no usable elevation in rows {win['grow0'] + r0}.. "
                                             f"(the first at lon {win['bounds'][0] + (cc + 0.5) / CELLS_DEG:.5f}, lat "
                                             f"{win['bounds'][3] - (r0 + rr + 0.5) / CELLS_DEG:.5f}: {why}). Nothing was written to Drive.")
                v = strip[good]
                if v.size:
                    st["n_all"] += int(v.size)
                    st["min_all"], st["max_all"] = min(st["min_all"], float(v.min())), max(st["max_all"], float(v.max()))
                vin = strip[inside & good]
                if vin.size:
                    st["n_in"] += int(vin.size)
                    st["sum_in"] += float(vin.sum(dtype="f8"))
                    st["min_in"], st["max_in"] = min(st["min_in"], float(vin.min())), max(st["max_in"], float(vin.max()))
                if CLIP_TO_POLYGON:
                    strip[~inside] = NODATA
                out.write(strip, 1, window=rasterio.windows.Window(0, r0, w, nr))
                st["crcs"][r0] = zlib.crc32(np.ascontiguousarray(strip))
                if (r0 // STRIP_ROWS) % 10 == 0 or r0 + nr >= h:
                    log.info("rows %d-%d of %d written | %.0f s", r0, r0 + nr, h, time.time() - t0)
            use = (st["n_in"], st["sum_in"] / max(1, st["n_in"]), st["min_in"], st["max_in"]) if st["n_in"] else \
                  (st["n_all"], None, st["min_all"], st["max_all"])
            out.update_tags(AREA_OR_POINT="Area", TIFFTAG_IMAGEDESCRIPTION=f"Copernicus DEM GLO-30, {ROI_LABEL} aquifer bounding box, native 1 arc-second grid",
                            source_product="Copernicus DEM GLO-30 (COG tiles Copernicus_DSM_COG_10_*_DEM), AWS Open Data copernicus-dem-30m",
                            cell_centres="on whole arc-seconds, as in the source tiles (the source declares them as points; GDAL places the edges half a cell away)",
                            vertical_reference="heights in metres above the EGM2008 geoid, as the Copernicus DEM documentation states",
                            clipped_to_polygon=str(CLIP_TO_POLYGON), generated=iso(utcnow()),
                            copyright="Produced using Copernicus WorldDEM-30 (c) DLR e.V. 2010-2014 and (c) Airbus Defence and Space GmbH 2014-2018 "
                                      "provided under COPERNICUS by the European Union and ESA; all rights reserved.")
            out.update_tags(1, STATISTICS_MINIMUM=str(use[2]), STATISTICS_MAXIMUM=str(use[3]), STATISTICS_VALID_PERCENT="100",
                            **({"STATISTICS_MEAN": str(use[1])} if use[1] is not None else {}))
            out.set_band_description(1, "elevation above the EGM2008 geoid (m)")
    finally:
        for hnd in handles.values():
            hnd.close()
    st["seconds"] = time.time() - t0
    log.info("Built %s: %d x %d cells, %s; aquifer cells %s, elevation %.1f to %.1f m, mean %s m", os.path.basename(path), w, h,
             human(os.path.getsize(path)), f"{st['n_in']:,}", st["min_in"], st["max_in"], f"{st['sum_in'] / st['n_in']:.1f}" if st["n_in"] else "-")
    return st

In [ ]:
# ═══════════════ CELL 7: check the closed file: header, every strip, and cells read again straight from the source tiles ═══════════════
def check_output(path, win, tiles, st, poly):
    """The finished file on scratch, before any byte of it goes to Drive:
      1  header: grid, size, type, no-data, compression exactly as built
      2  every strip read back from the closed file and compared with the checksum taken when it was written
      3  POINT_CHECKS random cells (the same cells every run) looked up by longitude / latitude in the output and in the source tile
         that holds them, by rasterio's own coordinate arithmetic (independent of the integer indexing used to build the file); the
         two values must be bit-identical (outside the polygon only when it was not clipped)."""
    with rasterio.open(path) as ds:
        if (ds.width, ds.height, ds.count, ds.dtypes[0], str(ds.crs)) != (win["w"], win["h"], 1, EXPECT_DTYPE, "EPSG:4326") \
                or not ds.transform.almost_equals(win["transform"], precision=1e-12) or ds.nodata != NODATA:
            raise DataIntegrityError(f"{path}: the header differs from what was built: {ds.width}x{ds.height} {ds.dtypes} {ds.crs} {ds.transform} nodata {ds.nodata}")
        comp = ds.compression
        t0 = time.time()
        for r0, want in st["crcs"].items():
            nr = min(STRIP_ROWS, win["h"] - r0)
            got = ds.read(1, window=rasterio.windows.Window(0, r0, win["w"], nr))
            if zlib.crc32(np.ascontiguousarray(got)) != want:
                raise DataIntegrityError(f"{os.path.basename(path)}: rows {r0}-{r0 + nr} read back differently from what was written.")
        log.info("Read-back: %d strips of the closed file identical to what was written (%.0f s), compression %s", len(st["crcs"]), time.time() - t0, comp)

        rng = np.random.default_rng(20261022)
        rows = rng.integers(0, win["h"], POINT_CHECKS)
        cols = rng.integers(0, win["w"], POINT_CHECKS)
        lon = -180 + (win["gcol0"] + cols) / CELLS_DEG                 # cell centres
        lat = 90 - (win["grow0"] + rows) / CELLS_DEG
        mine = np.array([v[0] for v in ds.sample(zip(lon, lat))], dtype="f4")
    inside = shapely.contains_xy(poly, lon, lat) if poly is not None else np.zeros(POINT_CHECKS, bool)
    by_tile, n_cmp, n_skip = {}, 0, 0
    for i in range(POINT_CHECKS):             # the tile that holds the cell (see Cell 3); the value is then looked up by coordinates in both files
        by_tile.setdefault((89 - (win["grow0"] + int(rows[i])) // CELLS_DEG, (win["gcol0"] + int(cols[i])) // CELLS_DEG - 180), []).append(i)
    tile_by = {(t["lat"], t["lon"]): t for t in tiles}
    for key, idx in by_tile.items():
        t = tile_by.get(key)
        if t is None:
            n_skip += len(idx)
            continue
        with rasterio.open(t["path"]) as src:
            theirs = np.array([v[0] for v in src.sample([(lon[i], lat[i]) for i in idx])], dtype="f4")
        for j, i in enumerate(idx):
            if CLIP_TO_POLYGON and not inside[i]:
                if mine[i] != NODATA:
                    raise DataIntegrityError(f"cell at {lon[i]:.5f}, {lat[i]:.5f} is outside the polygon but holds {mine[i]}, not NODATA")
                continue
            if mine[i].tobytes() != theirs[j].tobytes():
                raise DataIntegrityError(f"cell at {lon[i]:.5f}, {lat[i]:.5f}: the output holds {mine[i]!r}, the source tile {t['name']} {theirs[j]!r}")
            n_cmp += 1
    log.info("Point check: %d random cells identical to the source tiles (%d outside any tile or skipped)", n_cmp, n_skip)
    return dict(points_compared=n_cmp, compression=str(comp))

In [ ]:
# ═══════════════ CELL 8: run - six steps, each export step synchronised with Drive before the next begins ═══════════════
#   step 1  export  the tile manifest (what is about to be built, from which tiles)          -> Drive, synchronised
#   step 2  local   download and verify the tiles
#   step 3  local   build the GeoTIFF on scratch, check it, read it back, compare with the source tiles
#   step 4  export  the GeoTIFF                                                              -> Drive, synchronised
#   step 5  export  the manifest again, now with the checksum, statistics and checks of the output -> Drive, synchronised
#   step 6  final   flush_and_unmount, mount again, read every file of the run back from the fresh mount and compare
def manifest_doc(win, tiles, absent, status, out=None, st=None, checks=None):
    return json.dumps(dict(
        notebook="GLO_DEM_Ogallala.ipynb", created=iso(utcnow()), status=status, test_mode=TEST_MODE,
        source=dict(product="Copernicus DEM GLO-30 (Copernicus_DSM_COG_10 tiles)", base=AWS_BASE,
                    note="1 x 1 degree COGs of 3600 x 3600 float32 cells; ETag = MD5 of the tile"),
        window=dict(rows=win["h"], columns=win["w"], global_row0=win["grow0"], global_col0=win["gcol0"],
                    west=win["bounds"][0], south=win["bounds"][1], east=win["bounds"][2], north=win["bounds"][3], cell_degrees=1 / CELLS_DEG),
        settings=dict(clip_to_polygon=CLIP_TO_POLYGON, nodata=NODATA, compress="DEFLATE", zlevel=COMPRESS_LEVEL, predictor=3, block=BLOCK_PX),
        tiles=[{k: t[k] for k in ("name", "size", "etag", "last_modified")} for t in tiles], tiles_absent=absent,
        output=out, statistics=st, checks=checks,
        drive_steps=[dict(step=a, file=b, bytes=c, md5=d, seconds=e, flushed=f) for a, b, c, d, e, f in SYNC_LOG]), indent=1)


def log_projection(poly, win, tiles, st, out_bytes, t_download):
    """What the whole bounding box of the aquifer will cost, from the rehearsal."""
    full = make_window(poly, box=poly.bounds)
    n_tiles = len(tiles_for(full))
    rng = np.random.default_rng(1)
    w, s, e, n = poly.bounds
    frac = float(shapely.contains_xy(poly, rng.uniform(w, e, 20000), rng.uniform(s, n, 20000)).mean())
    cells = full["h"] * full["w"]
    per_cell = out_bytes / max(1, st["n_in"]) if CLIP_TO_POLYGON else out_bytes / (win["h"] * win["w"])
    size = per_cell * (frac * cells if CLIP_TO_POLYGON else cells)
    tile_mb = sum(t["size"] for t in tiles) / len(tiles) / 1e6
    secs = st["seconds"] * cells / (win["h"] * win["w"]) + (t_download / len(tiles)) * n_tiles
    log.info("PROJECTION for the whole bounding box: %d x %d cells (%s), %d tiles (about %.1f GB to download), about %s of them inside the "
             "aquifer; output about %s; about %.0f min at the speed of this rehearsal.", full["h"], full["w"], f"{cells:,}", n_tiles,
             n_tiles * tile_mb / 1000, f"{100 * frac:.0f} %", human(size), secs / 60)


def already_done(win, tiles):
    """True when Drive holds the finished file made from exactly these tiles (same names, sizes and ETags)."""
    mp, op = os.path.join(DRIVE_OUT, MANIFEST_NAME), os.path.join(DRIVE_OUT, OUT_NAME)
    if FORCE or not (os.path.exists(mp) and os.path.exists(op)):
        return False
    try:
        m = json.load(open(mp, encoding="utf-8"))
    except Exception:
        return False
    same = [(t["name"], t["size"], t["etag"]) for t in tiles] == [(t["name"], t["size"], t["etag"]) for t in m.get("tiles", [])]
    o = m.get("output") or {}
    return (m.get("status") == "complete" and same and m.get("settings", {}).get("clip_to_polygon") == CLIP_TO_POLYGON
            and m.get("window", {}).get("rows") == win["h"] and o.get("bytes") == os.path.getsize(op) and o.get("md5") == md5sum(op))


def run():
    t_run = time.time()
    SYNC_LOG.clear()
    TIMES.clear()
    TOTALS.clear()
    try:
        ensure_disk_free("start")
        clean_drive_leftovers()
        poly = load_roi()
        win = make_window(poly)
        tiles, absent = build_manifest(win, poly)
        if already_done(win, tiles):
            log.info("%s on Drive was made from these same tiles and its checksum matches - nothing to do (FORCE = True builds it again).", OUT_NAME)
            return dict(done=False, path=os.path.join(DRIVE_OUT, OUT_NAME))

        planned = manifest_doc(win, tiles, absent, "planned")
        md5 = drive_put_text(planned, MANIFEST_NAME, "step 1: tile manifest")                                            # step 1
        record_step("step 1: tile manifest", MANIFEST_NAME, md5, len(planned.encode("utf-8")))

        t_dl = time.time()
        tiles = download_all(tiles)                                                                                      # step 2
        t_dl = time.time() - t_dl
        local = os.path.join(BUILD_DIR, OUT_NAME)
        if os.path.exists(local):
            os.remove(local)
        st = build_output(local, win, tiles, poly)                                                                       # step 3
        checks = check_output(local, win, tiles, st, poly)
        out_info = dict(name=OUT_NAME, bytes=os.path.getsize(local), md5=md5sum(local))
        stats = dict(cells_inside_aquifer=st["n_in"], inside_min_m=st["min_in"], inside_max_m=st["max_in"],
                     inside_mean_m=(st["sum_in"] / st["n_in"]) if st["n_in"] else None, window_min_m=st["min_all"], window_max_m=st["max_all"],
                     cells_in_window=win["h"] * win["w"])
        if TEST_MODE:
            log_projection(poly, win, tiles, st, out_info["bytes"], t_dl)
        drive_put(local, os.path.join(DRIVE_OUT, OUT_NAME), "step 4: GeoTIFF")                                           # step 4
        record_step("step 4: GeoTIFF", OUT_NAME, out_info["md5"], out_info["bytes"])

        final = manifest_doc(win, tiles, absent, "complete", out_info, stats, checks)                                  # step 5
        md5m = drive_put_text(final, MANIFEST_NAME, "step 5: final manifest")
        record_step("step 5: final manifest", MANIFEST_NAME, md5m, len(final))

        hard_flush()                                                                                                     # step 6
        for name, size, want in ((OUT_NAME, out_info["bytes"], out_info["md5"]), (MANIFEST_NAME, len(final.encode("utf-8")), md5m)):
            p = os.path.join(DRIVE_OUT, name)
            if not _read_back_ok(p, size, want):
                raise PipelineStop(f"STEP 6: {name} on Drive does not read back identically after the final flush and fresh mount.")
        if not KEEP_TILES:
            shutil.rmtree(TILE_DIR, ignore_errors=True)
        os.remove(local)
        log.info("DONE: %s (%s) on Drive, %d export steps synchronised, final flush and re-read from a fresh mount passed, %.1f min",
                 OUT_NAME, human(out_info["bytes"]), len(SYNC_LOG), (time.time() - t_run) / 60)
        return dict(done=True, path=os.path.join(DRIVE_OUT, OUT_NAME), stats=stats, checks=checks)
    except PipelineStop as e:
        print("\n" + "=" * 78 + f"\n  STOPPED (the files on Drive are whole; re-run)\n  {e}\n" + "=" * 78 + "\n")
        return None
    except DataIntegrityError as e:
        print("\n" + "=" * 78 + f"\n  STOPPED: DATA CHECK FAILED - nothing of this build went to Drive\n  {e}\n" + "=" * 78 + "\n")
        return None
    finally:
        log.info("run time %.1f min | by stage (s): %s | downloaded %s", (time.time() - t_run) / 60, {k: round(v) for k, v in TIMES.items()},
                 human(TOTALS.get("bytes_downloaded", 0)))


RESULT = run()

In [ ]:
# ═══════════════ CELL 9: what is on Drive ═══════════════
def describe():
    mp, op = os.path.join(DRIVE_OUT, MANIFEST_NAME), os.path.join(DRIVE_OUT, OUT_NAME)
    if not (os.path.exists(mp) and os.path.exists(op)):
        print("No finished file on Drive yet.")
        return
    m = json.load(open(mp, encoding="utf-8"))
    print(f"{op}\n{human(os.path.getsize(op))} | status {m['status']} | {m['window']['rows']} x {m['window']['columns']} cells of 1/3600 degree | "
          f"lon {m['window']['west']:.5f} to {m['window']['east']:.5f}, lat {m['window']['south']:.5f} to {m['window']['north']:.5f}")
    with rasterio.open(op) as ds:
        print(f"{ds.crs} | {ds.dtypes[0]} | nodata {ds.nodata} | {ds.compression} | blocks {ds.block_shapes[0]} | tags {ds.tags(1)}")
        step = max(1, ds.height // 400)
        a = ds.read(1, out_shape=(ds.height // step, ds.width // step))        # decimated read: a look, not the statistics
        ok = a != ds.nodata
        print(f"decimated 1/{step}: {100 * ok.mean():.1f} % of cells hold a value, elevation {a[ok].min():.1f} to {a[ok].max():.1f} m, mean {a[ok].mean():.1f} m")
    s = m.get("statistics") or {}
    if s:
        print(f"inside the aquifer ({s['cells_inside_aquifer']:,} cells): {s['inside_min_m']:.1f} to {s['inside_max_m']:.1f} m, mean {s['inside_mean_m']:.1f} m "
              "(no cell at or below 0 m, no NaN, no NODATA)" if s.get("inside_mean_m") is not None else f"window: {s['window_min_m']} to {s['window_max_m']} m")
    print(f"checks: {m.get('checks')}")
    print(f"{len(m['tiles'])} tiles used; Drive export steps:")
    for st in m.get("drive_steps", []):
        print(f"   {st['step']:28s} {st['file']:46s} {human(st['bytes']):>9s} MD5 {st['md5'][:10]}... {st['seconds']:6.1f} s  flushed+remounted: {st['flushed']}")
    led = os.path.join(DRIVE_OUT, LEDGER_NAME)
    if os.path.exists(led):
        print(f"ledger {LEDGER_NAME}: {sum(1 for _ in open(led, encoding='utf-8'))} line(s)")


describe()

# Reading the file (it is 5-6 GB uncompressed, so read windows, never everything):
#   import rasterio
#   from rasterio.windows import from_bounds
#   with rasterio.open("/content/drive/MyDrive/MSUGWB/GLO_DEM/GLO30_DEM_Ogallala_1arcsec.tif") as ds:
#       win = from_bounds(-101.5, 38.0, -101.0, 38.5, ds.transform)
#       dem = ds.read(1, window=win, masked=True)         # masked=True hides the NODATA cells (there are none inside the aquifer)
# Heights are metres above the EGM2008 geoid. The grid is Copernicus' own 1 arc-second grid; nothing was resampled.